# Day 32 — Packaging & Infrastructure as Code

Day 30 built a RAG API. It runs *on your laptop*. Today: make it a **reproducible artifact**
(a container image) and describe the infrastructure it needs as **code** you can plan, apply,
and diff — so "what is running in prod" is a file in git, not tribal knowledge.

We can't run Docker or Terraform inside a notebook, so you'll **build the mental models from
scratch** — a layer-caching simulator and a 60-line declarative resource engine with
plan/apply/drift — and read the real `Dockerfile` / `main.tf` alongside.

## Learning objectives

1. Explain what a container image is (layers, content-addressing) and why layer order controls
   build speed.
2. Write a production `Dockerfile` for a Python LLM service: multi-stage, non-root, cached deps.
3. Explain the declarative model: desired state vs recorded state vs real state; plan = the diff.
4. Implement `plan` / `apply` / drift-detection for a tiny resource graph with dependencies.
5. Read a Terraform config for ECR + Lambda + API Gateway and say what each block does.
6. List what must *not* go in an image (secrets, config) and how it gets in at runtime.

## Agenda (60 min)

| # | Segment | Time |
| - | ------- | ---- |
| 0 | "Works on my machine" → artifact + declared infra | 4 min |
| 1 | The bash-script deploy, and watching it drift | 8 min |
| 2 | Container images from scratch: layers & caching | 13 min |
| 3 | A declarative infra engine: plan / apply / drift | 14 min |
| 4 | The real Terraform, block by block | 6 min |
| 5 | Config & secrets: the 12-factor cut that matters | 12 min |
| 6 | Bridge to CI/CD | 3 min |
| 7 | Exercises and self-check quiz | — |

## Setup

```bash
source ../../../.venv/bin/activate
```

Pure standard library. No Docker, Terraform, or cloud account needed.


## 0 — "Works on my machine" → artifact + declared infra (4 min)

Two problems, two tools:

| Problem | Symptom | Fix |
| ------- | ------- | --- |
| The app isn't reproducible | different Python, missing system lib, "did you `pip install`?" | **package it**: a container image — app + deps + runtime, one content-addressed blob |
| The infra isn't reproducible | someone clicked in a console; nobody knows the current state; staging ≠ prod | **declare it**: infrastructure as code — a file that *is* the desired state, applied by a tool |

Both share one idea: **the source of truth is a file in version control**, and a tool
converges reality to it. You review infra changes in a PR, exactly like code.

## 1 — The bash-script deploy, and watching it drift (8 min)

The pre-IaC way: a script that pushes code and pokes the environment.

In [1]:
# a stylised "deploy.sh", modelled as operations against a live environment dict
def deploy_v1(env):
    env["code_version"] = "v1"
    env.setdefault("instances", 2)
    env.setdefault("env_vars", {})["MODEL"] = "claude-haiku-4-5"
    return env

prod = {}
deploy_v1(prod)
print("after deploy:", prod)

# ...three weeks later, someone is firefighting at 2am and bumps capacity by hand:
prod["instances"] = 6
prod["env_vars"]["MODEL"] = "claude-sonnet-4-5"     # and quietly changes the model

# now re-run the SAME deploy script for a routine release:
deploy_v1(prod)
print("after re-deploy:", prod)
print("instances still 6? ->", prod["instances"], "  (script said 2, but only *sets* if unset)")
print("model reverted? ->", prod["env_vars"]["MODEL"])


after deploy: {'code_version': 'v1', 'instances': 2, 'env_vars': {'MODEL': 'claude-haiku-4-5'}}
after re-deploy: {'code_version': 'v1', 'instances': 6, 'env_vars': {'MODEL': 'claude-haiku-4-5'}}
instances still 6? -> 6   (script said 2, but only *sets* if unset)
model reverted? -> claude-haiku-4-5


The script is **imperative** — a list of steps, not a description of the goal. It doesn't
*converge*: whether the end state is right depends on the starting state. The 2am change is
invisible (no diff, no record), and the next deploy either stomps it or preserves it by
accident. Multiply across staging/prod/dev and environments drift apart.

Declarative IaC inverts this: you write the *desired* end state; the tool computes and shows
the steps (`plan`), then makes reality match (`apply`). Re-applying an unchanged config is a
no-op. A manual change shows up as **drift** on the next plan.

## 2 — Container images from scratch: layers & caching (13 min)

An image is a **stack of read-only layers**, each the filesystem *diff* produced by one build
instruction, identified by the hash of its contents. Containers add a thin writable layer on
top. Two images that share a base share those layers on disk and over the network.

The build cache rule: a layer is reused iff **its instruction AND every layer before it** are
unchanged. That's why instruction order is a performance decision.

In [2]:
import hashlib

def h(*parts):
    return hashlib.sha256("|".join(map(str, parts)).encode()).hexdigest()[:12]

def build(instructions, context, prev_cache):
    """instructions: list of (op, arg). context: {path: content-hash} the build can see.
       Returns (layers, new_cache, hits)."""
    layers, chain, hits = [], "scratch", 0
    for op, arg in instructions:
        # what this instruction depends on:
        if op == "COPY":
            dep = context.get(arg, "MISSING")
        else:
            dep = arg
        key = h(chain, op, dep)
        if key in prev_cache:
            hits += 1
        chain = key
        layers.append((op, arg, key))
    return layers, {k for *_, k in layers}, hits

instr_bad = [("FROM", "python:3.12-slim"),
             ("COPY", "src/"),              # <-- app code copied BEFORE deps
             ("RUN", "pip install -r requirements.txt"),
             ("CMD", "uvicorn app:app")]
instr_good = [("FROM", "python:3.12-slim"),
              ("COPY", "requirements.txt"), # <-- deps first; they change rarely
              ("RUN", "pip install -r requirements.txt"),
              ("COPY", "src/"),             # app code last; changes every commit
              ("CMD", "uvicorn app:app")]

ctx1 = {"requirements.txt": h("reqs-A"), "src/": h("code-A")}
_, cache_bad, _ = build(instr_bad, ctx1, set())
_, cache_good, _ = build(instr_good, ctx1, set())

ctx2 = {"requirements.txt": h("reqs-A"), "src/": h("code-B")}   # only the app code changed
_, _, hits_bad = build(instr_bad, ctx2, cache_bad)
_, _, hits_good = build(instr_good, ctx2, cache_good)
print(f"code-only change  ->  bad order: {hits_bad}/4 layers cached, "
      f"good order: {hits_good}/5 layers cached")
print("bad order re-runs `pip install` every commit; good order skips it")


code-only change  ->  bad order: 1/4 layers cached, good order: 3/5 layers cached
bad order re-runs `pip install` every commit; good order skips it


Now the real thing. A production `Dockerfile` for the RAG service — multi-stage (build tools
stay out of the final image), non-root, deps cached separately from code:

In [3]:
dockerfile = '''# ---- stage 1: build a clean venv --------------------------------------
FROM python:3.12-slim AS build
ENV PIP_NO_CACHE_DIR=1 PYTHONDONTWRITEBYTECODE=1
WORKDIR /app
RUN python -m venv /venv
ENV PATH="/venv/bin:$PATH"
COPY requirements.txt .
RUN pip install -r requirements.txt          # cached until requirements.txt changes

# ---- stage 2: runtime, no build tools -------------------------------
FROM python:3.12-slim AS runtime
RUN useradd --create-home --uid 10001 appuser
COPY --from=build /venv /venv
ENV PATH="/venv/bin:$PATH" PYTHONUNBUFFERED=1
WORKDIR /app
COPY --chown=appuser:appuser src/ ./src/     # last layer: changes every commit
USER appuser
EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=3s CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/healthz')"
CMD ["uvicorn", "src.app:app", "--host", "0.0.0.0", "--port", "8000"]
'''
dockerignore = "\n".join([".venv", "__pycache__", "*.pyc", ".git", ".env",
                           "tests", "*.ipynb", "notebooks"])
from pathlib import Path
Path("Dockerfile").write_text(dockerfile)
Path(".dockerignore").write_text(dockerignore + "\n")
print(dockerfile)
print("--- .dockerignore ---"); print(dockerignore)


# ---- stage 1: build a clean venv --------------------------------------
FROM python:3.12-slim AS build
ENV PIP_NO_CACHE_DIR=1 PYTHONDONTWRITEBYTECODE=1
WORKDIR /app
RUN python -m venv /venv
ENV PATH="/venv/bin:$PATH"
COPY requirements.txt .
RUN pip install -r requirements.txt          # cached until requirements.txt changes

# ---- stage 2: runtime, no build tools -------------------------------
FROM python:3.12-slim AS runtime
RUN useradd --create-home --uid 10001 appuser
COPY --from=build /venv /venv
ENV PATH="/venv/bin:$PATH" PYTHONUNBUFFERED=1
WORKDIR /app
COPY --chown=appuser:appuser src/ ./src/     # last layer: changes every commit
USER appuser
EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=3s CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/healthz')"
CMD ["uvicorn", "src.app:app", "--host", "0.0.0.0", "--port", "8000"]

--- .dockerignore ---
.venv
__pycache__
*.pyc
.git
.env
tests
*.ipynb
notebooks


Why each choice:

- **Multi-stage** — compilers, headers, pip cache live in `build` and never ship. Smaller
  image = faster pulls = faster cold starts = less attack surface.
- **`requirements.txt` before `src/`** — the §2 cache lesson: dep install is the slow layer;
  don't invalidate it on every code change.
- **Non-root `appuser`** — a container escape starts unprivileged.
- **`.dockerignore`** — keeps `.env`, `.git`, tests, notebooks out of the build context (and
  the image). A committed `.env` in an image is a leaked secret.
- **`HEALTHCHECK`** — the orchestrator needs a liveness signal to restart or drain.
- **Pinned base tag** — `python:3.12-slim`, not `latest`; ideally a digest in prod.

## 3 — A declarative infra engine: plan / apply / drift (14 min)

Terraform in ~60 lines. Resources have a type, a desired config, and dependencies. The engine
keeps a **state file** (what it last created). `plan` diffs desired vs state; `apply` executes
the plan and updates state; a **refresh** compares state vs the *real world* to find drift.

In [4]:
from dataclasses import dataclass, field
import json as _json

@dataclass
class Resource:
    type: str
    name: str
    config: dict
    depends_on: list = field(default_factory=list)
    @property
    def id(self): return f"{self.type}.{self.name}"

class Engine:
    def __init__(self, world):
        self.world = world                 # the "cloud": {id: config}
        self.state = {}                    # what we believe we created: {id: config}

    def _order(self, resources):           # topological sort on depends_on
        done, out = set(), []
        rs = {r.id: r for r in resources}
        def visit(r):
            if r.id in done: return
            for d in r.depends_on: visit(rs[d])
            done.add(r.id); out.append(r)
        for r in resources: visit(r)
        return out

    def plan(self, resources):
        actions = []
        desired_ids = {r.id for r in resources}
        for r in self._order(resources):
            if r.id not in self.state:
                actions.append(("create", r.id, r.config))
            elif self.state[r.id] != r.config:
                actions.append(("update", r.id,
                                {k: r.config[k] for k in r.config
                                 if r.config.get(k) != self.state[r.id].get(k)}))
        for sid in self.state:
            if sid not in desired_ids:
                actions.append(("destroy", sid, {}))
        return actions

    def apply(self, resources):
        for act, rid, cfg in self.plan(resources):
            if act in ("create", "update"):
                base = dict(self.state.get(rid, {}))
                base.update(next(r.config for r in resources if r.id == rid))
                self.world[rid] = base
                self.state[rid] = dict(base)
            elif act == "destroy":
                self.world.pop(rid, None); self.state.pop(rid, None)
        return "apply complete"

    def drift(self):
        return {rid: {"expected": self.state[rid], "actual": self.world.get(rid)}
                for rid in self.state if self.world.get(rid) != self.state[rid]}

world = {}
eng = Engine(world)

repo   = Resource("ecr_repo", "api", {"name": "rag-api", "scan_on_push": True})
fn     = Resource("lambda", "api", {"image": "rag-api:git-abc123", "memory_mb": 1024,
                                    "env": {"MODEL": "claude-haiku-4-5"}},
                  depends_on=["ecr_repo.api"])
gw     = Resource("api_gateway", "api", {"route": "POST /ask", "target": "lambda.api"},
                  depends_on=["lambda.api"])
stack = [repo, fn, gw]

print("PLAN #1 (empty world):")
for a in eng.plan(stack): print("  ", a[0], a[1], a[2])
eng.apply(stack)
print("PLAN #2 (no changes):", eng.plan(stack))       # [] -> converged


PLAN #1 (empty world):
   create ecr_repo.api {'name': 'rag-api', 'scan_on_push': True}
   create lambda.api {'image': 'rag-api:git-abc123', 'memory_mb': 1024, 'env': {'MODEL': 'claude-haiku-4-5'}}
   create api_gateway.api {'route': 'POST /ask', 'target': 'lambda.api'}
PLAN #2 (no changes): []


In [5]:
# bump memory + model in the config (a normal PR):
fn.config["memory_mb"] = 2048
fn.config["env"] = {"MODEL": "claude-sonnet-4-5"}
print("PLAN after edit:")
for a in eng.plan(stack): print("  ", a)
eng.apply(stack)

# someone changes memory in the console by hand -> drift:
world["lambda.api"]["memory_mb"] = 512
print("\nDRIFT:", _json.dumps(eng.drift(), indent=2))
print("\nnext plan re-asserts desired state:")
for a in eng.plan(stack): print("  ", a)   # update lambda.api back to 2048


PLAN after edit:
   ('update', 'lambda.api', {'memory_mb': 2048, 'env': {'MODEL': 'claude-sonnet-4-5'}})

DRIFT: {
  "lambda.api": {
    "expected": {
      "image": "rag-api:git-abc123",
      "memory_mb": 2048,
      "env": {
        "MODEL": "claude-sonnet-4-5"
      }
    },
    "actual": {
      "image": "rag-api:git-abc123",
      "memory_mb": 512,
      "env": {
        "MODEL": "claude-sonnet-4-5"
      }
    }
  }
}

next plan re-asserts desired state:


That's the whole model: **desired (config) → recorded (state) → real (world)**. Plan is the
diff between desired and recorded; drift is the diff between recorded and real. Re-apply is
idempotent. Destroying a resource = removing it from the config.

## 4 — The real Terraform, block by block (6 min)

In [6]:
terraform = '''terraform {
  required_providers { aws = { source = "hashicorp/aws", version = "~> 5.0" } }
  backend "s3" {                     # remote state, locked, shared by the team
    bucket = "acme-tfstate"
    key    = "rag-api/terraform.tfstate"
    region = "us-east-1"
    use_lockfile = true
  }
}

variable "image_tag" { type = string }            # set by CI to the git SHA

resource "aws_ecr_repository" "api" {
  name                 = "rag-api"
  image_scanning_configuration { scan_on_push = true }
}

resource "aws_lambda_function" "api" {
  function_name = "rag-api"
  package_type  = "Image"
  image_uri     = "${aws_ecr_repository.api.repository_url}:${var.image_tag}"
  memory_size   = 2048
  timeout       = 30
  environment { variables = { MODEL = "claude-haiku-4-5" } }   # NOT secrets
}

resource "aws_apigatewayv2_api" "api" {
  name          = "rag-api"
  protocol_type = "HTTP"
}

resource "aws_apigatewayv2_integration" "api" {
  api_id                 = aws_apigatewayv2_api.api.id
  integration_type       = "AWS_PROXY"
  integration_uri        = aws_lambda_function.api.invoke_arn
  payload_format_version = "2.0"
}

resource "aws_apigatewayv2_route" "ask" {
  api_id    = aws_apigatewayv2_api.api.id
  route_key = "POST /ask"
  target    = "integrations/${aws_apigatewayv2_integration.api.id}"
}

output "endpoint" { value = aws_apigatewayv2_api.api.api_endpoint }
'''
from pathlib import Path
Path("main.tf").write_text(terraform)
print(terraform)


terraform {
  required_providers { aws = { source = "hashicorp/aws", version = "~> 5.0" } }
  backend "s3" {                     # remote state, locked, shared by the team
    bucket = "acme-tfstate"
    key    = "rag-api/terraform.tfstate"
    region = "us-east-1"
    use_lockfile = true
  }
}

variable "image_tag" { type = string }            # set by CI to the git SHA

resource "aws_ecr_repository" "api" {
  name                 = "rag-api"
  image_scanning_configuration { scan_on_push = true }
}

resource "aws_lambda_function" "api" {
  function_name = "rag-api"
  package_type  = "Image"
  image_uri     = "${aws_ecr_repository.api.repository_url}:${var.image_tag}"
  memory_size   = 2048
  timeout       = 30
  environment { variables = { MODEL = "claude-haiku-4-5" } }   # NOT secrets
}

resource "aws_apigatewayv2_api" "api" {
  name          = "rag-api"
  protocol_type = "HTTP"
}

resource "aws_apigatewayv2_integration" "api" {
  api_id                 = aws_apigatewayv2_api.api.id


- **`terraform` / `backend "s3"`** — state lives in S3 with a lock so two engineers can't
  `apply` at once. Never keep state only on a laptop.
- **`${aws_ecr_repository.api.repository_url}`** — an interpolation creates an implicit
  `depends_on`; Terraform builds the same DAG our `_order()` did.
- **`var.image_tag`** — CI passes the git SHA (immutable tag); a release is
  `terraform apply -var image_tag=$SHA`.
- **`environment.variables`** — non-secret config only. Secrets come from a secrets manager at
  runtime (§5).
- **`terraform plan`** in a PR shows the diff for review; `apply` runs post-merge.

## 5 — Config & secrets: the 12-factor cut that matters (12 min)

The [12-factor](https://12factor.net) points that bite LLM services specifically:

**III. Config in the environment.** Model name, `MAX_TOKENS`, index URL, feature flags — env
vars, not baked into the image. One image, promoted unchanged dev → staging → prod; only the
config differs. Our Dockerfile has zero config values in it.

**Secrets are not config.** `ANTHROPIC_API_KEY`, DB passwords — never in the image, never in
`main.tf`, never in env vars visible in a console. They live in a secrets manager and are
injected at container start:

In [7]:
# the pattern: app reads secrets from the environment; a launcher populates it from a vault
class FakeSecretsManager:
    _store = {"prod/rag-api/ANTHROPIC_API_KEY": "sk-ant-REDACTED-not-real",
              "prod/rag-api/DB_PASSWORD": "hunter2"}
    @classmethod
    def get(cls, name): return cls._store[name]

def launch_container(image, config_env, secret_refs):
    env = dict(config_env)                             # non-secret config
    for var, ref in secret_refs.items():
        env[var] = FakeSecretsManager.get(ref)         # resolved at start, in memory only
    redacted = {k: (v[:6] + "..." if "KEY" in k or "PASSWORD" in k else v) for k, v in env.items()}
    return {"image": image, "effective_env": redacted}

print(launch_container(
    image="rag-api:git-abc123",
    config_env={"MODEL": "claude-haiku-4-5", "MAX_TOKENS": "1024", "INDEX_URI": "s3://kb/v7"},
    secret_refs={"ANTHROPIC_API_KEY": "prod/rag-api/ANTHROPIC_API_KEY",
                 "DB_PASSWORD": "prod/rag-api/DB_PASSWORD"}))


{'image': 'rag-api:git-abc123', 'effective_env': {'MODEL': 'claude-haiku-4-5', 'MAX_TOKENS': '1024', 'INDEX_URI': 's3://kb/v7', 'ANTHROPIC_API_KEY': 'sk-ant...', 'DB_PASSWORD': 'hunter...'}}


**IX. Disposability — graceful shutdown.** An LLM request can be mid-stream when the
orchestrator sends `SIGTERM` (deploy, scale-in, spot reclaim). Drain instead of dropping it:

In [8]:
import signal, time

class GracefulServer:
    def __init__(self): self.inflight = 0; self.accepting = True
    def handle(self):
        if not self.accepting: raise RuntimeError("503 draining")
        self.inflight += 1
        try: time.sleep(0.001); return "answer"
        finally: self.inflight -= 1
    def on_sigterm(self):
        self.accepting = False                 # stop taking new work; LB pulls us out
        deadline = time.time() + 30            # grace period
        while self.inflight and time.time() < deadline:
            time.sleep(0.05)
        return f"shutdown: {self.inflight} requests dropped"

s = GracefulServer()
s.handle()
print(s.on_sigterm())      # 0 dropped — nothing in flight


shutdown: 0 requests dropped


Other points worth a line: **X. dev/prod parity** — same image, same backing services
(a real vector DB in dev, not an in-memory stub, or you'll ship bugs); **XI. logs to stdout**
— the platform collects them, the app doesn't manage files; **immutable image tags** — deploy
`git-abc123`, never `latest`, so a rollback is exact.

**Image hygiene for the pipeline (Day 33):** scan on push (CV->fail the build), generate an
SBOM, sign the image, and only deploy tags that passed.

**Where this goes next:** Day 33 — the pipeline that builds this image, gates it on the eval
suite, and rolls it out (canary / rollback) without a 2am bash session.

## Exercises

1. **Layer budget.** Extend the §2 `build()` simulator to also track an approximate *size* per
   layer (`FROM`=120, `RUN pip`=200, `COPY src/`=5, else 1). Compare final image size and
   cache hit-rate for the "bad" vs "good" instruction order across 5 successive code-only
   commits.
2. **`terraform destroy`.** Add a `destroy_all()` to `Engine` that tears the stack down in
   *reverse* dependency order. Show it removes `api_gateway` before `lambda` before `ecr_repo`.
3. **Targeted drift repair.** Add `plan(resources, target="lambda.api")` that limits the plan
   to one resource and its dependencies — the `-target` flag. Use it to fix the §3 drift
   without touching the gateway.
4. **A real Dockerfile review.** Take the `instr_bad` order, write it as an actual Dockerfile,
   and list every line you'd change for production (order, base tag, user, healthcheck,
   `.dockerignore`). Justify each.
5. **Secret leak hunt.** Given a `Dockerfile` with `ENV ANTHROPIC_API_KEY=sk-ant-...` and a
   `COPY . .`, name all the places that key now lives (layers, image history, registry, CI
   logs) and how you'd remediate after it's already pushed.
6. **Module-ise it.** Sketch how you'd turn `main.tf` into a reusable `rag_service` module
   with inputs (`image_tag`, `memory`, `model`, `env_name`) so staging and prod are two
   4-line `module` blocks. What stays in the module, what becomes an input?

## Self-check quiz

1. What *is* a container image layer, and what determines whether the build cache reuses it?
2. Why does `COPY requirements.txt` before `COPY src/` make builds faster?
3. Define desired state, recorded state (state file), and real state. Which two does `plan`
   compare? Which two reveal drift?
4. Why is re-running `terraform apply` on an unchanged config safe, but re-running a deploy
   bash script often isn't?
5. Name three things that must never be in an image and where each belongs instead.
6. What does a multi-stage build buy you over a single stage?
7. Your orchestrator sends `SIGTERM` during a deploy. What should a well-behaved LLM service
   do in the next 30 seconds?
